In [18]:
import numpy as np
import pandas as pd

In [19]:
# makes sure we are using proper data types
def validate_data(df):
    df = df.copy()
    
    df["season"] = df["season"].astype(int)
    df["date"] = pd.to_datetime(df["date"])
    df["score1"] = df["score1"].astype(int)
    df["score2"] = df["score2"].astype(int)
    df["result"] = df["result"].astype(int)
    return df


def create_team_rows(df, league="NFL"):

    home = pd.DataFrame({
        "league": league,
        "season": df["season"],
        "team_id": df["team1"],
        "points_for": df["score1"],
        "points_against": df["score2"],
        "result": df["result"],
        "is_home": 1  
    })

    result_from_away_perspective = {3: 0, 0: 3, 1: 1}
    away = pd.DataFrame({
        "league": league,
        "season": df["season"],
        "team_id": df["team2"],
        "points_for": df["score2"],
        "points_against": df["score1"],
        "result": df["result"].map(result_from_away_perspective),
        "is_home": 0 
    })

    return pd.concat([home, away], ignore_index = True)

def get_standing(rows):

    team_rows = rows.copy()
    team_rows["wins"] = (team_rows["result"] == 3).astype(int)
    team_rows["losses"] = (team_rows["result"] == 0).astype(int)
    team_rows["ties"] = (team_rows["result"] == 1).astype(int)

    standings = team_rows.groupby(["league", "season", "team_id"], as_index=False).agg({
        "result": "count",
        "wins": "sum",
        "losses": "sum", 
        "ties": "sum",
        "points_for": "sum",
        "points_against": "sum"
    }).rename(columns={"result": "games_played"})

    standings["win_pct"] = (standings["wins"] + 0.5 * standings["ties"]) / standings["games_played"]
    standings["point_diff"] = standings["points_for"] - standings["points_against"]

    standings = standings.sort_values(["league", "season", "win_pct", "point_diff", "team_id"],
        ascending=[True, True, False, False, True])
    standings["rank"] = standings.groupby(["league", "season"]).cumcount() + 1

    columns = ["league", "season", "team_id", "games_played", "wins", "losses", 
              "ties", "win_pct", "points_for", "points_against", "point_diff", "rank"]
    
    return standings[columns]

def get_rankings(standings, output = "data/us_leagues/csv", filename="nfl_standings.csv"):
    path = f"{output}/{filename}"
    standings_sorted = standings.sort_values(["season", "rank"])
    standings_sorted.to_csv(path, index=False)
    seasons = standings["season"].unique()
    total_teams = len(standings)
    
    print(f"Exported: {path}")
    print(f"Total rows: {total_teams}")
    
    return path


In [20]:
# run to generate nfl_standings.csv
nfl_data = pd.read_csv("../csv/nfl_data.csv")
nfl_clean = validate_data(nfl_data)
team_rows = create_team_rows(nfl_clean, league="NFL")
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../csv", filename = "nfl_standings.csv")

Exported: ../csv/nfl_standings.csv
Total rows: 1738


In [21]:
# run to generate mlb_standings.csv
mlb_data = pd.read_csv("../csv/mlb_data.csv")
mlb_clean = validate_data(mlb_data)
team_rows = create_team_rows(mlb_clean, league="MLB")
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../csv", filename = "mlb_standings.csv")

Exported: ../csv/mlb_standings.csv
Total rows: 1288


In [22]:
# run to generate nba_standings.csv
nba_data = pd.read_csv("../csv/nba_data.csv")
nba_clean = validate_data(nba_data)
team_rows = create_team_rows(nba_clean, league="NBA")
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../csv", filename = "nba_standings.csv")

Exported: ../csv/nba_standings.csv
Total rows: 1662
